# 02 - Unir poblacion municipal (CONAPO) y calcular tasas
Insumos:
- data/processed/suicidio_chihuahua_2019_2024.csv (de 01_filter_chihuahua.ipynb)
- data/raw/2_Gran_Gedad_08_CH.xlsx (CONAPO, poblacion por municipio/sexo/anio)

Denominador: poblacion TOTAL (ambos sexos) por municipio y anio.
Numerador: casos de suicidio por Mun_resid y anio (residencia habitual,
ver docs/methodology.md).

In [ ]:
import pandas as pd


## 1. Cargar poblacion CONAPO y agregar por municipio-anio (sumar sexos)

In [ ]:
df_pob_raw = pd.read_excel('../data/raw/2_Gran_Gedad_08_CH.xlsx')
print(f'Filas crudas (por municipio-anio-sexo): {len(df_pob_raw):,}')
df_pob_raw.head()


In [ ]:
# Extraer codigo de municipio de 3 digitos desde CLAVE (ej. 8001 -> '001')
# CLAVE = CLAVE_ENT * 1000 + numero_municipio
df_pob_raw['mun_codigo'] = (df_pob_raw['CLAVE'] - df_pob_raw['CLAVE_ENT'] * 1000).astype(int).astype(str).str.zfill(3)

# Sumar HOMBRES + MUJERES para obtener poblacion total por municipio-anio
df_pob = (
    df_pob_raw
    .groupby(['mun_codigo', 'NOM_MUN', 'AÑO'], as_index=False)['POB_TOTAL']
    .sum()
    .rename(columns={'AÑO': 'anio', 'POB_TOTAL': 'poblacion_total'})
)
print(f'Filas agregadas (por municipio-anio): {len(df_pob):,}')
df_pob.head()


## 2. Verificacion rapida: poblacion total de Chihuahua por anio
Comparar contra cifras conocidas (ej. Ciudad Juarez ~1.6 millones en 2024,
segun fuentes revisadas en la conversacion con Claude).

In [ ]:
pob_estatal_por_anio = df_pob.groupby('anio')['poblacion_total'].sum()
print(pob_estatal_por_anio.loc[2019:2024])
print()
juarez_2024 = df_pob[(df_pob['NOM_MUN'].str.contains('Ju', case=False)) & (df_pob['anio'] == 2024)]
juarez_2024


## 3. Cargar casos de suicidio y agregar por municipio-anio
Usa Mun_resid (residencia habitual), consistente con la poblacion CONAPO
que tambien es por residencia.

In [ ]:
df_chih = pd.read_csv('../data/processed/suicidio_chihuahua_2019_2024.csv', encoding='utf-8', low_memory=False, dtype=str)

casos_por_municipio_anio = (
    df_chih
    .groupby(['Mun_resid', 'anio_dataset'])
    .size()
    .reset_index(name='casos')
    .rename(columns={'Mun_resid': 'mun_codigo', 'anio_dataset': 'anio'})
)
casos_por_municipio_anio['anio'] = casos_por_municipio_anio['anio'].astype(int)
print(f'Combinaciones municipio-anio con al menos 1 caso: {len(casos_por_municipio_anio):,}')
casos_por_municipio_anio.head()


## 4. Unir poblacion y casos, calcular tasa
IMPORTANTE: usar 'left' desde la poblacion (no desde los casos), para que
los municipios SIN ningun caso en un anio queden con casos=0, no desaparezcan
de la tabla. Un municipio ausente en casos_por_municipio_anio no es un dato
faltante, es 0 casos reales.

In [ ]:
df_tasas = df_pob[df_pob['anio'].between(2019, 2024)].merge(
    casos_por_municipio_anio, on=['mun_codigo', 'anio'], how='left'
)
df_tasas['casos'] = df_tasas['casos'].fillna(0).astype(int)
df_tasas['tasa_por_100k'] = round(df_tasas['casos'] / df_tasas['poblacion_total'] * 100000, 2)

print(f'Filas finales (municipio x anio, 2019-2024): {len(df_tasas):,}')
df_tasas.sort_values('tasa_por_100k', ascending=False).head(15)


## 5. Verificacion: municipios sin match (revisar codigos antes de continuar)
Si algun Mun_resid de los casos NO aparece en la poblacion CONAPO, esos casos
se pierden silenciosamente del merge. Verificar aqui antes de seguir.

In [ ]:
codigos_casos = set(casos_por_municipio_anio['mun_codigo'])
codigos_poblacion = set(df_pob['mun_codigo'])
sin_match = codigos_casos - codigos_poblacion
print(f'Codigos de municipio en casos SIN match en poblacion CONAPO: {sin_match}')
if sin_match:
    registros_afectados = casos_por_municipio_anio[casos_por_municipio_anio['mun_codigo'].isin(sin_match)]
    print(f'Casos afectados (se perderian en el merge): {registros_afectados["casos"].sum()}')
    registros_afectados


## 6. Guardar dataset final con tasas

In [ ]:
df_tasas.to_csv('../data/processed/tasas_suicidio_municipal_chihuahua_2019_2024.csv', index=False, encoding='utf-8')
print(f'Guardado: {len(df_tasas):,} filas (municipio x anio)')


## 7. Hallazgos
_Documentar aqui: si hubo codigos sin match (seccion 5) y como se resolvieron,
que municipios muestran las tasas mas altas/inestables, y si es evidente la
necesidad de suavizado o agregacion plurianual (ver decision pendiente en
README.md) antes de seguir al analisis espacial.